In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.deterministic import DeterministicProcess, CalendarFourier
from statsmodels.tsa.seasonal import seasonal_decompose
from scipy.signal import periodogram
from sklearn.model_selection import train_test_split
from statsmodels.graphics.tsaplots import plot_pacf
from xgboost import XGBRegressor

In [47]:
df = pd.read_csv (
    "teste/dados_sabesp_cantareira_historico.csv", sep=';', parse_dates=['Data'], dayfirst=True
)

In [48]:
df.head()

,Data,Chuva (mm),Volume Útil Armazenado (%),Volume Útil Armazenado (hm³),Volume Total Armazenado (hm³),Vazão Captada (m³/s),Vazão Produzida (m³/s),Vazão Jusante (m³/s),Vazão Natural (m³/s),Vazão Afluente (m³/s),Variação do Volume Útil (%),Chuva Acumulada no Mês (mm),Chuva Média Historica (mm),Vazão Jusante no mês (m³/s),Vazão Jusante Média Histórica (m³/s),Vazão Natural no Mês (m³/s),Vazão Natural Media Historica (m³/s),Vazão Produzida no Mês (m³/s),Vazão Retirada no Mês (m³/s)
0,2000-01-01,"30,9","47,11254","365,50555","1072,05098","29,0",NaN,"10,16","70,885","70,885","0,32635","30,9","261,51618","10,16","22,55533","70,885",NaN,NaN,"29,0"
1,2000-01-02,"29,1","47,77519","370,64646","1077,1919","28,8",NaN,"10,18","94,864","94,864","0,66265","60,0","261,51618","10,17","22,55533","82,875",NaN,NaN,"28,9"
2,2000-01-03,"35,175","48,59438","377,00187","1083,5473","33,3",NaN,"7,29","108,208","108,208","0,81919","95,175","261,51618","9,21","22,55533","91,319",NaN,NaN,"30,36667"
3,2000-01-04,"18,725","49,9968","387,88206","1094,42749","32,4",NaN,"6,34","162,936","162,936","1,40242","113,9","261,51618","8,4925","22,55533","109,223",NaN,NaN,"30,875"
4,2000-01-05,"25,9","51,94578","403,00251","1109,54794","32,9",NaN,"6,33","211,604","211,604","1,94898","139,8","261,51618","8,06","22,55533","129,699",NaN,NaN,"31,28"


In [49]:
dfv = df[['Data', 'Volume Útil Armazenado (%)']].copy()
dfv = dfv.rename(columns= {'Volume Útil Armazenado (%)':'Volume_util'})
dfv['Data'] = pd.to_datetime(dfv['Data'])
dfv = dfv.set_index('Data')
dfv['Volume_util'] = dfv['Volume_util'].str.strip()
dfv['Volume_util'] = dfv['Volume_util'].str.replace(',','.', regex=False)
dfv['Volume_util'] = pd.to_numeric(dfv['Volume_util'])
dfv.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 9412 entries, 2000-01-01 to 2025-10-07
Data columns (total 1 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Volume_util  9412 non-null   float64
dtypes: float64(1)
memory usage: 147.1 KB


In [50]:
dfv.head()

,Volume_util
Data,
2000-01-01,47.11254
2000-01-02,47.77519
2000-01-03,48.59438
2000-01-04,49.99680
2000-01-05,51.94578


In [51]:
dfv['Time'] = np.arange(len(dfv.index))
dfv.head()

,Volume_util,Time
Data,,
2000-01-01,47.11254,0
2000-01-02,47.77519,1
2000-01-03,48.59438,2
2000-01-04,49.99680,3
2000-01-05,51.94578,4


In [52]:
df_periodo = dfv.loc['2016-01-01':'2025-12-31'].copy()
df_semanal = df_periodo.resample('W').mean()
df_semanal['Time'] = np.arange(len(df_semanal.index))
df_semanal.head()

,Volume_util,Time
Data,,
2016-01-03,1.112363,0
2016-01-10,2.530224,1
2016-01-17,5.531559,2
2016-01-24,12.642724,3
2016-01-31,15.129069,4


### Tendência da série temporal do volume útil do sistema cantareira

#### Tendência com a regressão linear

In [ ]:
X = df_semanal.loc[:, ['Time']]
y = df_semanal.loc[:, ['Volume_util']]

model = LinearRegression()
model.fit(X,y)

y_pred = pd.Series(model.predict(X).flatten(), index=X.index)
df_semanal['y_pred'] = y_pred

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")

plt.rc(

    "figure",
    autolayout = True, 
    figsize = (11,4),
    titlesize = 18,
    titleweight = 'bold',

)

plt.rc(
    "axes",
    labelweight = 'bold',
    labelsize = "large",
    titleweight = "bold",
    titlesize = 16,
    titlepad = 10
    )

fig, ax = plt.subplots(figsize=(16,5))

ax.plot(df_semanal['Time'], df_semanal['Volume_util'], data=df_semanal, color= '0.75')
ax.plot (df_semanal['Time'], df_semanal['y_pred'], color='tab:blue', linewidth=3, label='Regressão Linear', zorder=3)

ax.set_title('Volume útil (%) do sistema cantareira entre 2000 e 2025')
ax.set_xlabel('Time')
ax.set_ylabel('Volume_util')

#### Visualizando a tendência por médias móveis

In [ ]:
moving_average = df_semanal['Volume_util'].rolling(
    window = 365, 
    center = True, 
    min_periods = 182, 

).mean()

ax = df_semanal['Volume_util'].plot (style="-", color="0.5", figsize=(15,5))
moving_average.plot(
    ax=ax, linewidth=3, title = "Volume útil do sistema cantareira em médias móveis", legend=False
)

#### Tendência + Sazonalidade usando termos de Fourier 

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")
plt.rc("figure", autolayout=True, figsize=(11, 5))
plt.rc(
    "axes",
    labelweight="bold",
    labelsize="large",
    titleweight="bold",
    titlesize=16,
    titlepad=10,
)

plot_params = dict(
    color="0.75",
    style=".-",
    markeredgecolor="0.25",
    markerfacecolor="0.25",
    legend=False,
)

%config InlineBackend.figure_format = 'retina'

def seasonal_plot (X, y, period, freq, ax=None):
    if ax is None:
        _, ax = plt.subplots()
    palette = sns.color_palette("husl", n_colors = X[period].nunique(),)
    ax = sns.lineplot(
        x=freq,
        y=y,
        hue=period,
        data=X,
        ci=False,
        ax=ax,
        palette=palette,
        legend=False,
    )
    ax.set_title(f"Seasonal Plot ({period}/{freq})")
    for line, name in zip(ax.lines, X[period].unique()):
        y_ = line.get_ydata()[-1]
        ax.annotate(
            name,
            xy=(1, y_),
            xytext=(6, 0),
            color=line.get_color(),
            xycoords=ax.get_yaxis_transform(),
            textcoords="offset points",
            size=14,
            va="center",
        )

    return ax 

def plot_periodogram(ts, detrend='linear', ax=None):
    from scipy.signal import periodogram
    fs=pd.Timedelta("365D") / pd.Timedelta("1D")
    frequencies, spectrum = periodogram(
        ts,
        fs=fs,
        detrend=detrend,
        window="boxcar",
        scaling='spectrum',
    )
    if ax is None:
        _, ax = plt.subplots()
    ax.step(frequencies, spectrum, color="purple")
    ax.set_xscale("log")
    ax.set_xticks([1, 2, 4, 6, 12, 26, 52, 104])
    ax.set_xticklabels(
        [
            
            "Annual (1)",
            "Semiannual (2)",
            "Quarterly (4)",
            "Bimonthly (6)",
            "Monthly (12)",
            "Biweekly (26)",
            "Weekly (52)",
            "Semiweekly (104)",
        ],
        rotation=30,
    )

    ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
    ax.set_ylabel("Variance")
    ax.set_title("Periodogram")
    return ax

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
import matplotlib.pyplot as plt

decomposicao = seasonal_decompose(df_semanal['Volume_util'], model='additive', period=52)

fig, axes = plt.subplots(4, 1, sharex=True, figsize=(12, 10))


decomposicao.observed.plot(ax=axes[0], color='blue', legend=False)
axes[0].set_ylabel('Observado')


decomposicao.trend.plot(ax=axes[1], color='firebrick', legend=False)
axes[1].set_ylabel('Tendência')


decomposicao.seasonal.plot(ax=axes[2], color='green', legend=False)
axes[2].set_ylabel('Sazonalidade')


decomposicao.resid.plot(ax=axes[3], color='gray', legend=False, style='.')
axes[3].set_ylabel('Resíduo')


fig.suptitle('Decomposição da Série Temporal - Sistema Cantareira', fontsize=16)
plt.xlabel('Ano')
plt.tight_layout(rect=[0, 0.03, 1, 0.97]) # Ajusta o layout para o título principal não sobrepor os gráficos
plt.show()

In [ ]:
plot_periodogram(df_semanal.Volume_util);

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")
plt.rc("figure", autolayout=True, figsize=(11, 4))
plt.rc(
    "axes",
    labelweight="bold",
    labelsize="large",
    titleweight="bold",
    titlesize=16,
    titlepad=10,
)
plot_params = dict(
    color="0.75",
    style="-.",
    markeredgecolor="0.25",
    markerfacecolor="0.25",
)
%config InlineBackend.figure_format = 'retina'


def lagplot(x, y=None, lag=1, standardize=False, ax=None, **kwargs):
    from matplotlib.offsetbox import AnchoredText
    x_ = x.shift(lag)
    if standardize:
        x_ = (x_ - x_.mean()) / x_.std()
    if y is not None:
        y_ = (y - y.mean()) / y.std() if standardize else y
    else:
        y_ = x
    corr = y_.corr(x_)
    if ax is None:
        fig, ax = plt.subplots()
    scatter_kws = dict(
        alpha=0.75,
        s=3,
    )
    line_kws = dict(color='C3', )
    ax = sns.regplot(x=x_,
                     y=y_,
                     scatter_kws=scatter_kws,
                     line_kws=line_kws,
                     lowess=True,
                     ax=ax,
                     **kwargs)
    at = AnchoredText(
        f"{corr:.2f}",
        prop=dict(size="large"),
        frameon=True,
        loc="upper left",
    )
    at.patch.set_boxstyle("square, pad=0.0")
    ax.add_artist(at)
    ax.set(title=f"Lag {lag}", xlabel=x_.name, ylabel=y_.name)
    return ax


def plot_lags(x, y=None, lags=6, nrows=1, lagplot_kwargs={}, **kwargs):
    import math
    kwargs.setdefault('nrows', nrows)
    kwargs.setdefault('ncols', math.ceil(lags / nrows))
    kwargs.setdefault('figsize', (kwargs['ncols'] * 2, nrows * 2 + 0.5))
    fig, axs = plt.subplots(sharex=True, sharey=True, squeeze=False, **kwargs)
    for ax, k in zip(fig.get_axes(), range(kwargs['nrows'] * kwargs['ncols'])):
        if k + 1 <= lags:
            ax = lagplot(x, y, lag=k + 1, ax=ax, **lagplot_kwargs)
            ax.set_title(f"Lag {k + 1}", fontdict=dict(fontsize=14))
            ax.set(xlabel="", ylabel="")
        else:
            ax.axis('off')
    plt.setp(axs[-1, :], xlabel=x.name)
    plt.setp(axs[:, 0], ylabel=y.name if y is not None else x.name)
    fig.tight_layout(w_pad=0.1, h_pad=0.1)
    return fig


In [ ]:
_ = plot_lags(df_semanal.Volume_util, lags=12, nrows=2)
_ = plot_pacf(df_semanal.Volume_util, lags=12)

#### Considerando apenas o periodo entre 2016 e 2023

In [ ]:


# --- 1. AGREGAÇÃO: Transformar Dados Diários em Médias Semanais ---

# Garantir que a frequência seja diária antes de qualquer coisa
dfv = dfv.asfreq('D') 

# Filtrar para o período de interesse (2016-2025)
df_periodo = dfv.loc['2016-01-01':'2025-12-31'].copy()

# AQUI ESTÁ A MUDANÇA PRINCIPAL: Reamostrar para frequência semanal ('W'), calculando a média
# O resultado é um novo DataFrame onde cada linha representa a média de uma semana.
df_semanal = df_periodo.resample('W').mean()

# A série alvo agora é a série semanal.
# Vamos interpolar para preencher qualquer semana que possa ter ficado com NaN.
y1 = df_semanal["Volume_util"].interpolate(method='linear')

# --- 2. Preparação das Features para a Frequência Semanal ---

fourier = CalendarFourier(freq="A", order=4)
dp = DeterministicProcess(
    index=y1.index,  # Usando o novo índice semanal
    constant=True,
    order=1,
    # REMOVIDO: seasonal=True, pois não se aplica a dados semanais
    additional_terms=[fourier],  # Sazonalidade anual ainda é importante
    drop=True,
)
x1 = dp.in_sample()

# A data de corte continua a mesma
split_date = '2024-01-01'

# Divisão dos dados semanais em treino e teste
X_train1 = x1.loc[x1.index < split_date]
X_test1 = x1.loc[x1.index >= split_date]
y_train1 = y1.loc[y1.index < split_date]
y_test1 = y1.loc[y1.index >= split_date]

# --- 3. Modelo 1: Regressão Linear na Série Semanal ---

model1 = LinearRegression()
model1.fit(X_train1, y_train1)

y_pred_train1 = pd.Series(model1.predict(X_train1), index=X_train1.index)
y_pred_test1 = pd.Series(model1.predict(X_test1), index=X_test1.index)

# --- 4. Modelo 2: XGBoost nos Resíduos Semanais ---

# Calcular resíduos
y_res_train1 = y_train1 - y_pred_train1 
y_res_test1 = y_test1 - y_pred_test1
y_res1 = pd.concat([y_res_train1, y_res_test1])

# Criar lags (agora, lag=1 significa 1 semana atrás)
def make_lags(ts, lags):
    return pd.concat(
        {f'y_lag_{i}': ts.shift(i) for i in range(1, lags + 1)},
        axis=1)
x2 = make_lags(y_res1, lags=2)
x2 = x2.fillna(0.0)

# Dividir dados para o XGBoost
X2_train1 = x2.loc[x2.index < split_date]
X2_test1 = x2.loc[x2.index >= split_date]

# Treinar XGBoost
model2 = XGBRegressor(n_estimators=50, learning_rate=0.05, max_depth=5, random_state=0) 
model2.fit(X2_train1, y_res_train1)

# Previsão dos resíduos no treino
y_pred_res_train1 = pd.Series(model2.predict(X2_train1), index=X2_train1.index)

# Previsão RECURSIVA para os resíduos no teste (a lógica é a mesma, mas agora passo-a-passo semanal)
X2_test_recursive1 = X2_test1.copy()
y_pred_res_test_list1 = []
for index in X2_test_recursive1.index:
    features_hoje = X2_test_recursive1.loc[[index]]
    pred_hoje = model2.predict(features_hoje)[0]
    y_pred_res_test_list1.append(pred_hoje)
    
    try:
        proximo_index = X2_test_recursive1.index[X2_test_recursive1.index.get_loc(index) + 1]
        X2_test_recursive1.loc[proximo_index, 'y_lag_1'] = pred_hoje
        if 'y_lag_2' in X2_test_recursive1.columns:
            X2_test_recursive1.loc[proximo_index, 'y_lag_2'] = features_hoje['y_lag_1'].values[0]
    except IndexError:
        break
y_pred_res_test1 = pd.Series(y_pred_res_test_list1, index=X2_test1.index)

# --- 5. Previsão Híbrida Final (Semanal) ---

y_final_train1 = y_pred_train1 + y_pred_res_train1
y_final_test1 = y_pred_test1 + y_pred_res_test1

# --- 6. Visualização Final (Gráfico de Médias Semanais) ---

fig, ax = plt.subplots(figsize=(14, 7))

y1.plot(ax=ax, color='black', alpha=0.7, style='-', label='Dados Reais (Média Semanal)')
y_final_train1.plot(ax=ax, color='green', linestyle='--', label='Previsão Híbrida Semanal (Treino)')
y_final_test1.plot(ax=ax, color='red', linestyle='-', label='Previsão Híbrida Semanal (Teste)')

ax.set_title('Previsão Híbrida de Médias Semanais (2016-2025)')
ax.set_ylabel('Volume Médio Semanal (%)')
ax.legend()
plt.show()

In [ ]:
# Importar as funções de métricas do scikit-learn
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# --- 7. Avaliação Numérica do Modelo (no Período de Teste) ---

# Garantir que não haja valores ausentes que possam causar erro
# (y_test e y_final_test devem ter o mesmo tamanho e estar alinhados)
y_test_clean1 = y_test1.dropna()
y_final_test_clean1 = y_final_test1[y_test_clean1.index]

# Calcular o MAE (Mean Absolute Error)
mae = mean_absolute_error(y_test_clean1, y_final_test_clean1)

# Calcular o RMSE (Root Mean Squared Error)
rmse = np.sqrt(mean_squared_error(y_test_clean1, y_final_test_clean1))

# Imprimir os resultados de forma clara
print("---" * 15)
print("   Avaliação do Modelo Híbrido no Período de Teste (2024-2025)")
print("---" * 15)
print(f"MAE (Erro Médio Absoluto): {mae:.2f}")
print(f"RMSE (Raiz do Erro Quadrático Médio): {rmse:.2f}\n")


#### Análise com a variável exógena de chuva

Através das análises de dados feitas até aqui, nóta-se um forte relação ciclica, causal entre a chuva e o volume útil do sistema cantareira, quando há periodos de chuva, sequentemente há periodos em que o volume do reservatório aumenta. Por isso, afim de tentar melhorar a previsão da volume que é nossa variável alvo, realizaremos a previsão da chuva

In [53]:
df.head()

,Data,Chuva (mm),Volume Útil Armazenado (%),Volume Útil Armazenado (hm³),Volume Total Armazenado (hm³),Vazão Captada (m³/s),Vazão Produzida (m³/s),Vazão Jusante (m³/s),Vazão Natural (m³/s),Vazão Afluente (m³/s),Variação do Volume Útil (%),Chuva Acumulada no Mês (mm),Chuva Média Historica (mm),Vazão Jusante no mês (m³/s),Vazão Jusante Média Histórica (m³/s),Vazão Natural no Mês (m³/s),Vazão Natural Media Historica (m³/s),Vazão Produzida no Mês (m³/s),Vazão Retirada no Mês (m³/s)
0,2000-01-01,"30,9","47,11254","365,50555","1072,05098","29,0",NaN,"10,16","70,885","70,885","0,32635","30,9","261,51618","10,16","22,55533","70,885",NaN,NaN,"29,0"
1,2000-01-02,"29,1","47,77519","370,64646","1077,1919","28,8",NaN,"10,18","94,864","94,864","0,66265","60,0","261,51618","10,17","22,55533","82,875",NaN,NaN,"28,9"
2,2000-01-03,"35,175","48,59438","377,00187","1083,5473","33,3",NaN,"7,29","108,208","108,208","0,81919","95,175","261,51618","9,21","22,55533","91,319",NaN,NaN,"30,36667"
3,2000-01-04,"18,725","49,9968","387,88206","1094,42749","32,4",NaN,"6,34","162,936","162,936","1,40242","113,9","261,51618","8,4925","22,55533","109,223",NaN,NaN,"30,875"
4,2000-01-05,"25,9","51,94578","403,00251","1109,54794","32,9",NaN,"6,33","211,604","211,604","1,94898","139,8","261,51618","8,06","22,55533","129,699",NaN,NaN,"31,28"


In [54]:
df_chuva = df[['Data', 'Chuva (mm)']].copy()
df_chuva = df_chuva.rename(columns= {'Chuva (mm)':'chuva'})
df_chuva['Data'] = pd.to_datetime(df_chuva['Data'])
df_chuva = df_chuva.set_index('Data')
df_chuva.head()

,chuva
Data,
2000-01-01,"30,9"
2000-01-02,"29,1"
2000-01-03,"35,175"
2000-01-04,"18,725"
2000-01-05,"25,9"


In [55]:
df_chuva['chuva'] = df_chuva['chuva'].str.strip()
df_chuva['chuva'] = df_chuva['chuva'].str.replace(',','.', regex=False)
df_chuva['chuva'] = pd.to_numeric(df_chuva['chuva'])

In [56]:
df_chuva['Time'] = np.arange(len(df_chuva.index))
df_chuva.head()

,chuva,Time
Data,,
2000-01-01,30.900,0
2000-01-02,29.100,1
2000-01-03,35.175,2
2000-01-04,18.725,3
2000-01-05,25.900,4


In [57]:
df_chuva = df_chuva.loc['2016-01-01':'2025-12-31'].copy()
df_semanal_chuva = df_chuva.resample('W').sum()

In [58]:
df_semanal_chuva['Time'] = np.arange(len(df_semanal_chuva.index))
df_semanal_chuva.head()

,chuva,Time
Data,,
2016-01-03,31.750,0
2016-01-10,19.825,1
2016-01-17,128.500,2
2016-01-24,0.575,3
2016-01-31,67.700,4


In [59]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.deterministic import DeterministicProcess, CalendarFourier
from xgboost import XGBRegressor

# --- 1. PREPARAÇÃO DOS DADOS DE CHUVA ---
# (Assumindo que 'df_chuva' já foi carregado e pré-processado como no seu script)

# Filtrar período e agregar para semanas
df_chuva = df_chuva.loc['2016-01-01':'2025-12-31'].copy()
df_semanal_chuva = df_chuva.resample('W').sum()

# Vamos garantir que não haja NaNs
df_semanal_chuva['chuva'] = df_semanal_chuva['chuva'].fillna(0.0)

# --- 2. ENGENHARIA DE FEATURES (X) PARA O MODELO DE CHUVA ---

# Sazonalidade (Fourier) e Tendência (Linear)
fourier_chuva = CalendarFourier(freq="A", order=4)
dp_chuva = DeterministicProcess(
    index=df_semanal_chuva.index,
    constant=True,
    order=1,
    additional_terms=[fourier_chuva],
    drop=True,
)
X_chuva_time = dp_chuva.in_sample()

# Lags da própria chuva (usaremos 8 semanas de histórico)
def make_lags(ts, lags, name='y'):
    return pd.concat(
        {f'{name}_lag_{i}': ts.shift(i) for i in range(1, lags + 1)},
        axis=1)

X_chuva_lags = make_lags(df_semanal_chuva['chuva'], lags=8, name='chuva')

# Combinar features de tempo e de lags
X_chuva_final = pd.concat([X_chuva_time, X_chuva_lags], axis=1)


# --- 3. CRIAÇÃO DOS ALVOS (Y) PARA A ESTRATÉGIA DIRETA ---
# Precisamos prever 4 semanas à frente

# y_t1 = chuva da próxima semana (shift -1)
y_chuva_t1 = df_semanal_chuva['chuva'].shift(-1).rename('chuva_t+1')
# y_t2 = chuva daqui a 2 semanas (shift -2)
y_chuva_t2 = df_semanal_chuva['chuva'].shift(-2).rename('chuva_t+2')
# y_t3 = chuva daqui a 3 semanas (shift -3)
y_chuva_t3 = df_semanal_chuva['chuva'].shift(-3).rename('chuva_t+3')
# y_t4 = chuva daqui a 4 semanas (shift -4)
y_chuva_t4 = df_semanal_chuva['chuva'].shift(-4).rename('chuva_t+4')

# Combinar todos os alvos em um único DataFrame
y_chuva_final = pd.concat([y_chuva_t1, y_chuva_t2, y_chuva_t3, y_chuva_t4], axis=1)


# --- 4. MONTAR DATASET DE TREINAMENTO E FAZER A DIVISÃO ---

# Juntar features (X) e alvos (y)
df_model_data_chuva = pd.concat([X_chuva_final, y_chuva_final], axis=1)

# Remover quaisquer linhas com NaN
# (no início, por causa dos lags; no fim, por causa dos shifts dos alvos)
df_model_data_chuva = df_model_data_chuva.dropna()

# Definir data de corte
split_date = '2024-01-01'

# Separar os dados de TREINO (usaremos para treinar os 4 modelos)
df_train_chuva = df_model_data_chuva.loc[df_model_data_chuva.index < split_date]

# Separar features de treino e alvos de treino
target_cols = ['chuva_t+1', 'chuva_t+2', 'chuva_t+3', 'chuva_t+4']
X_train_chuva = df_train_chuva.drop(columns=target_cols)
y_train_chuva = df_train_chuva[target_cols]

# Separar as FEATURES do período de TESTE (onde faremos as previsões)
# Note: Usamos o X_chuva_final original, que contém os dados de 2024-2025
X_test_chuva = X_chuva_final.loc[X_chuva_final.index >= split_date].copy()


# --- 5. TREINAR 4 MODELOS DE CHUVA (ESTRATÉGIA DIRETA) ---

print("Iniciando treinamento dos modelos de previsão de chuva...")
models_chuva = {} # Dicionário para guardar os modelos

for target in target_cols:
    print(f"Treinando modelo para {target}...")
    
    # Criar um novo modelo XGBoost para cada alvo
    model = XGBRegressor(
        n_estimators=100, # A chuva pode precisar de um modelo mais forte
        learning_rate=0.05,
        max_depth=5,
        random_state=0
    )
    
    # Treinar o modelo (ex: X_train_chuva -> y_train_chuva['chuva_t+1'])
    model.fit(X_train_chuva, y_train_chuva[target])
    
    # Guardar o modelo treinado
    models_chuva[target] = model

print("Treinamento dos modelos de chuva concluído.")


# --- 6. GERAR AS PREVISÕES DE CHUVA PARA 2024-2025 ---

# Dicionário para guardar as séries de previsões
predictions_chuva = {}

for target, model in models_chuva.items():
    print(f"Gerando previsões para {target}...")
    
    # Prever no conjunto de features de teste
    pred = model.predict(X_test_chuva)
    
    # Converter para Série do Pandas com o índice de datas correto
    predictions_chuva[target] = pd.Series(pred, index=X_test_chuva.index)

# Criar o DataFrame final com as previsões de chuva
df_chuva_forecasts = pd.DataFrame(predictions_chuva)

# A chuva não pode ser negativa
df_chuva_forecasts[df_chuva_forecasts < 0] = 0

print("\n--- Previsões de Chuva Geradas (Head) ---")
print(df_chuva_forecasts.head())

# --- 7. (CORRIGIDO) AVALIAÇÃO DOS MODELOS DE CHUVA ---

print("\n---" * 15)
print("   Avaliação dos Modelos de Previsão de Chuva (Período de Teste)")
print("---" * 15)

# 1. Obter os dados reais (ground truth) para o período de teste.
y_test_chuva_truth = y_chuva_final.loc[X_test_chuva.index]

# 2. Calcular métricas para cada um dos 4 modelos
for target in target_cols:
    
    # Seleciona a coluna de previsão (ex: 'chuva_t+1')
    preds_full = df_chuva_forecasts[target]
    
    # Seleciona a coluna de valores reais correspondente
    truth_full = y_test_chuva_truth[target]
    
    # ----- INÍCIO DA CORREÇÃO -----
    
    # 1. Remove os NaNs da série de valores reais (o 'truth' é quem tem os NaNs)
    truth_clean = truth_full.dropna()
    
    # 2. Filtra a série de previsões para que ela tenha O MESMO ÍNDICE
    #    da série de valores reais limpa. Isso garante alinhamento.
    preds_clean = preds_full.loc[truth_clean.index]
    
    # ----- FIM DA CORREÇÃO -----

    # Calcular MAE e RMSE usando as séries limpas e alinhadas
    mae = mean_absolute_error(truth_clean, preds_clean)
    rmse = np.sqrt(mean_squared_error(truth_clean, preds_clean))
    
    # Calcular a média real da chuva para dar contexto ao erro
    mean_truth = truth_clean.mean()
    
    # Imprimir os resultados para este modelo
    print(f"\nModelo: {target} (Previsão para {target[-1]} semana(s) à frente)")
    print(f"  Média Real da Chuva: {mean_truth:.2f} mm")
    print(f"  MAE (Erro Médio Absoluto): {mae:.2f} ")
    print(f"  RMSE (Raiz do Erro Quadrático Médio): {rmse:.2f} ")

print("---" * 15)

# O restante do código (print do head) continua igual

/home/jefferson_correia/tcc/venv/lib/python3.12/site-packages/statsmodels/tsa/deterministic.py:569: FutureWarning: 'A' is deprecated and will be removed in a future version, please use 'YE' instead.
  index = pd.date_range("2020-01-01", freq=freq, periods=1)


Iniciando treinamento dos modelos de previsão de chuva...
Treinando modelo para chuva_t+1...
Treinando modelo para chuva_t+2...
Treinando modelo para chuva_t+3...
Treinando modelo para chuva_t+4...
Treinamento dos modelos de chuva concluído.
Gerando previsões para chuva_t+1...
Gerando previsões para chuva_t+2...
Gerando previsões para chuva_t+3...
Gerando previsões para chuva_t+4...

--- Previsões de Chuva Geradas (Head) ---
            chuva_t+1  chuva_t+2  chuva_t+3  chuva_t+4
Data                                                  
2024-01-07  48.350739  50.531418  77.171143  65.765907
2024-01-14  45.096935  67.451340  74.416756  49.454937
2024-01-21  52.497570  73.520294  48.350750  53.632809
2024-01-28  64.214233  36.853298  35.285007  40.488922
2024-02-04  48.209869  64.809097  26.451813  79.624275

---
---
---
---
---
---
---
---
---
---
---
---
---
---
---
   Avaliação dos Modelos de Previsão de Chuva (Período de Teste)
---------------------------------------------

Modelo: chuva

In [60]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.deterministic import DeterministicProcess, CalendarFourier
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor # <-- Importa a nova ferramenta

# --- 1. PREPARAÇÃO DOS DADOS DE CHUVA ---
# (Esta seção não muda)
df_chuva = df_chuva.loc['2016-01-01':'2025-12-31'].copy()
df_semanal_chuva = df_chuva.resample('W').sum()
df_semanal_chuva['chuva'] = df_semanal_chuva['chuva'].fillna(0.0)

# --- 2. ENGENHARIA DE FEATURES (X) PARA O MODELO DE CHUVA ---
# (Esta seção não muda)
fourier_chuva = CalendarFourier(freq="A", order=4)
dp_chuva = DeterministicProcess(
    index=df_semanal_chuva.index,
    constant=True,
    order=1,
    additional_terms=[fourier_chuva],
    drop=True,
)
X_chuva_time = dp_chuva.in_sample()

def make_lags(ts, lags, name='y'):
    return pd.concat(
        {f'{name}_lag_{i}': ts.shift(i) for i in range(1, lags + 1)},
        axis=1)

X_chuva_lags = make_lags(df_semanal_chuva['chuva'], lags=8, name='chuva')
X_chuva_final = pd.concat([X_chuva_time, X_chuva_lags], axis=1)

# --- 3. CRIAÇÃO DOS ALVOS (Y) PARA A ESTRATÉGIA DIRETA ---
# (Esta seção não muda)
y_chuva_t1 = df_semanal_chuva['chuva'].shift(-1).rename('chuva_t+1')
y_chuva_t2 = df_semanal_chuva['chuva'].shift(-2).rename('chuva_t+2')
y_chuva_t3 = df_semanal_chuva['chuva'].shift(-3).rename('chuva_t+3')
y_chuva_t4 = df_semanal_chuva['chuva'].shift(-4).rename('chuva_t+4')
y_chuva_final = pd.concat([y_chuva_t1, y_chuva_t2, y_chuva_t3, y_chuva_t4], axis=1)

# --- 4. MONTAR DATASET DE TREINAMENTO E FAZER A DIVISÃO ---
# (Esta seção não muda)
df_model_data_chuva = pd.concat([X_chuva_final, y_chuva_final], axis=1)
df_model_data_chuva = df_model_data_chuva.dropna()

split_date = '2024-01-01'
df_train_chuva = df_model_data_chuva.loc[df_model_data_chuva.index < split_date]

target_cols = ['chuva_t+1', 'chuva_t+2', 'chuva_t+3', 'chuva_t+4']
X_train_chuva = df_train_chuva.drop(columns=target_cols)
y_train_chuva = df_train_chuva[target_cols]

X_test_chuva = X_chuva_final.loc[X_chuva_final.index >= split_date].copy()


# --- 5. (NOVO) TREINAR 1 MODELO DE CHUVA (ESTRATÉGIA MIMO) ---

print("Iniciando treinamento do modelo MultiOutput (MIMO) para chuva...")

# 1. Define o modelo base que será usado para cada saída
base_estimator = XGBRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=5,
    random_state=0
)

# 2. Envolve o modelo base com o MultiOutputRegressor
mimo_model_chuva = MultiOutputRegressor(base_estimator)

# 3. Treina o modelo UMA ÚNICA VEZ.
#    Ele vai treinar um XGBoost separado para cada alvo (t+1, t+2, t+3, t+4)
#    internamente, mas tudo gerenciado em um único objeto.
mimo_model_chuva.fit(X_train_chuva, y_train_chuva)

print("Treinamento do modelo MIMO concluído.")


# --- 6. (NOVO) GERAR AS PREVISÕES DE CHUVA PARA 2024-2025 ---

print("Gerando previsões MIMO para chuva...")

# 1. Faz a previsão UMA ÚNICA VEZ.
#    O resultado será um array NumPy com 4 colunas.
predictions_array = mimo_model_chuva.predict(X_test_chuva)

# 2. Converte o array em um DataFrame com o índice e colunas corretos.
df_chuva_forecasts = pd.DataFrame(
    predictions_array,
    index=X_test_chuva.index,
    columns=target_cols # Usa os nomes dos alvos
)

# 3. Garante que a chuva não seja negativa
df_chuva_forecasts[df_chuva_forecasts < 0] = 0

print("Previsões de chuva geradas.")


# --- 7. AVALIAÇÃO DOS MODELOS DE CHUVA ---
# (Esta seção não muda, pois `df_chuva_forecasts` tem o mesmo formato de antes)

print("\n---" * 15)
print("   Avaliação do Modelo MIMO de Previsão de Chuva (Período de Teste)")
print("---" * 15)

y_test_chuva_truth = y_chuva_final.loc[X_test_chuva.index]

for target in target_cols:
    preds_full = df_chuva_forecasts[target]
    truth_full = y_test_chuva_truth[target]
    
    # Remove NaNs do 'truth' (final da série) e alinha as previsões
    truth_clean = truth_full.dropna()
    preds_clean = preds_full.loc[truth_clean.index]

    mae = mean_absolute_error(truth_clean, preds_clean)
    rmse = np.sqrt(mean_squared_error(truth_clean, preds_clean))
    mean_truth = truth_clean.mean()
    
    print(f"\nHorizonte de Previsão: {target}")
    print(f"  Média Real da Chuva: {mean_truth:.2f} mm")
    print(f"  MAE (Erro Médio Absoluto): {mae:.2f} mm")
    print(f"  RMSE (Raiz do Erro Quadrático Médio): {rmse:.2f} mm")

print("---" * 15)

print("\n--- Amostra das Previsões de Chuva Geradas (Head) ---")
print(df_chuva_forecasts.head())

/home/jefferson_correia/tcc/venv/lib/python3.12/site-packages/statsmodels/tsa/deterministic.py:569: FutureWarning: 'A' is deprecated and will be removed in a future version, please use 'YE' instead.
  index = pd.date_range("2020-01-01", freq=freq, periods=1)


Iniciando treinamento do modelo MultiOutput (MIMO) para chuva...
Treinamento do modelo MIMO concluído.
Gerando previsões MIMO para chuva...
Previsões de chuva geradas.

---
---
---
---
---
---
---
---
---
---
---
---
---
---
---
   Avaliação do Modelo MIMO de Previsão de Chuva (Período de Teste)
---------------------------------------------

Horizonte de Previsão: chuva_t+1
  Média Real da Chuva: 22.99 mm
  MAE (Erro Médio Absoluto): 22.88 mm
  RMSE (Raiz do Erro Quadrático Médio): 28.90 mm

Horizonte de Previsão: chuva_t+2
  Média Real da Chuva: 22.41 mm
  MAE (Erro Médio Absoluto): 21.17 mm
  RMSE (Raiz do Erro Quadrático Médio): 27.67 mm

Horizonte de Previsão: chuva_t+3
  Média Real da Chuva: 21.72 mm
  MAE (Erro Médio Absoluto): 21.17 mm
  RMSE (Raiz do Erro Quadrático Médio): 28.61 mm

Horizonte de Previsão: chuva_t+4
  Média Real da Chuva: 21.21 mm
  MAE (Erro Médio Absoluto): 22.30 mm
  RMSE (Raiz do Erro Quadrático Médio): 29.90 mm
---------------------------------------------

In [61]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.multioutput import MultiOutputRegressor
from statsmodels.tsa.deterministic import DeterministicProcess, CalendarFourier
from xgboost import XGBRegressor

# --- 1. PREPARAÇÃO E ENGENHARIA DE FEATURES/ALVOS ---
# (Esta parte é idêntica às nossas tentativas anteriores)

print("Passo 1/7: Preparando dados, features e alvos...")
df_chuva = df_chuva.loc['2016-01-01':'2025-12-31'].copy()
df_semanal_chuva = df_chuva.resample('W').sum()
df_semanal_chuva['chuva'] = df_semanal_chuva['chuva'].fillna(0.0)

# Features (X)
fourier_chuva = CalendarFourier(freq="A", order=4)
dp_chuva = DeterministicProcess(
    index=df_semanal_chuva.index,
    constant=True, order=1,
    additional_terms=[fourier_chuva], drop=True,
)
X_chuva_time = dp_chuva.in_sample()
X_chuva_lags = make_lags(df_semanal_chuva['chuva'], lags=8, name='chuva')
X_chuva_final = pd.concat([X_chuva_time, X_chuva_lags], axis=1)

# Alvos (y)
y_chuva_t1 = df_semanal_chuva['chuva'].shift(-1).rename('chuva_t+1')
y_chuva_t2 = df_semanal_chuva['chuva'].shift(-2).rename('chuva_t+2')
y_chuva_t3 = df_semanal_chuva['chuva'].shift(-3).rename('chuva_t+3')
y_chuva_t4 = df_semanal_chuva['chuva'].shift(-4).rename('chuva_t+4')
y_chuva_final = pd.concat([y_chuva_t1, y_chuva_t2, y_chuva_t3, y_chuva_t4], axis=1)

# --- 2. DIVISÃO DOS DADOS ---
print("Passo 2/7: Dividindo dados em treino/teste...")

df_model_data_chuva = pd.concat([X_chuva_final, y_chuva_final], axis=1)
df_model_data_chuva = df_model_data_chuva.dropna()

split_date = '2024-01-01'
df_train_chuva = df_model_data_chuva.loc[df_model_data_chuva.index < split_date]

target_cols = ['chuva_t+1', 'chuva_t+2', 'chuva_t+3', 'chuva_t+4']
X_train = df_train_chuva.drop(columns=target_cols)
y_train = df_train_chuva[target_cols] # Alvo real (para LR)

# Features do conjunto de teste (para a previsão final)
X_test = X_chuva_final.loc[X_chuva_final.index >= split_date].copy()
# Alvos reais do conjunto de teste (para avaliação)
y_test_truth = y_chuva_final.loc[X_test.index]


# --- 3. TREINAR MODELO 1 (BASE: REGRESSÃO LINEAR MIMO) ---
print("Passo 3/7: Treinando Modelo 1 (Regressão Linear)...")

# Envolvemos a Regressão Linear no MultiOutputRegressor
model1_lr = MultiOutputRegressor(LinearRegression())
model1_lr.fit(X_train, y_train)

# --- 4. CALCULAR RESÍDUOS DO TREINO ---
print("Passo 4/7: Calculando resíduos do Modelo 1...")

# Prever no conjunto de TREINO para ver o que o LR errou
y_pred_lr_train = model1_lr.predict(X_train)

# Converter o array de previsão em um DataFrame para subtrair
y_pred_lr_train_df = pd.DataFrame(
    y_pred_lr_train,
    index=y_train.index,
    columns=y_train.columns
)

# Resíduos = Real - Previsão_LR. Este é o novo alvo do XGBoost
y_res_train = y_train - y_pred_lr_train_df


# --- 5. TREINAR MODELO 2 (RESÍDUOS: XGBOOST DIRETO) ---
print("Passo 5/7: Treinando Modelo 2 (XGBoost para Resíduos)...")

models_xgb_res = {} # Dicionário para guardar os 4 modelos de resíduos

for target in target_cols:
    print(f"Treinando modelo de resíduos para {target}...")
    
    # O alvo é a coluna de resíduo correspondente (ex: 'chuva_t+1')
    target_residual = y_res_train[target] 
    
    model = XGBRegressor(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=5,
        random_state=0
    )
    
    # O XGBoost é treinado com as MESMAS FEATURES (X_train)
    # mas para prever o RESÍDUO (target_residual)
    model.fit(X_train, target_residual)
    
    models_xgb_res[target] = model

print("Treinamento do Modelo 2 concluído.")


# --- 6. GERAR PREVISÕES FINAIS HÍBRIDAS ---
print("Passo 6/7: Gerando previsões finais híbridas...")

# 1. Previsão do Modelo 1 (Base LR) no conjunto de teste
forecast_lr_array = model1_lr.predict(X_test)
forecast_lr_df = pd.DataFrame(
    forecast_lr_array,
    index=X_test.index,
    columns=target_cols
)

# 2. Previsão do Modelo 2 (Resíduos XGB) no conjunto de teste
forecast_res_dict = {}
for target, model in models_xgb_res.items():
    pred_res = model.predict(X_test)
    forecast_res_dict[target] = pred_res
forecast_res_df = pd.DataFrame(forecast_res_dict, index=X_test.index)

# 3. Previsão Final = Previsão_LR + Previsão_Resíduo
df_chuva_forecasts = forecast_lr_df + forecast_res_df

# Garante que a chuva não seja negativa
df_chuva_forecasts[df_chuva_forecasts < 0] = 0

print("Previsões híbridas geradas.")


# --- 7. AVALIAÇÃO DO MODELO HÍBRIDO ---
print("Passo 7/7: Avaliando o modelo Híbrido-Direto...")
print("\n---" * 15)
print("   Avaliação do Modelo HÍBRIDO-DIRETO de Chuva (Período de Teste)")
print("---" * 15)

for target in target_cols:
    preds_full = df_chuva_forecasts[target]
    truth_full = y_test_truth[target]
    
    truth_clean = truth_full.dropna()
    preds_clean = preds_full.loc[truth_clean.index]

    mae = mean_absolute_error(truth_clean, preds_clean)
    rmse = np.sqrt(mean_squared_error(truth_clean, preds_clean))
    mean_truth = truth_clean.mean()
    
    print(f"\nHorizonte de Previsão: {target}")
    print(f"  Média Real da Chuva: {mean_truth:.2f} mm")
    print(f"  MAE (Erro Médio Absoluto): {mae:.2f} mm")
    print(f"  RMSE (Raiz do Erro Quadrático Médio): {rmse:.2f} mm")

print("---" * 15)

print("\n--- Amostra das Previsões de Chuva Geradas (Head) ---")
print(df_chuva_forecasts.head())

Passo 1/7: Preparando dados, features e alvos...
Passo 2/7: Dividindo dados em treino/teste...
Passo 3/7: Treinando Modelo 1 (Regressão Linear)...
Passo 4/7: Calculando resíduos do Modelo 1...
Passo 5/7: Treinando Modelo 2 (XGBoost para Resíduos)...
Treinando modelo de resíduos para chuva_t+1...


/home/jefferson_correia/tcc/venv/lib/python3.12/site-packages/statsmodels/tsa/deterministic.py:569: FutureWarning: 'A' is deprecated and will be removed in a future version, please use 'YE' instead.
  index = pd.date_range("2020-01-01", freq=freq, periods=1)


Treinando modelo de resíduos para chuva_t+2...
Treinando modelo de resíduos para chuva_t+3...
Treinando modelo de resíduos para chuva_t+4...
Treinamento do Modelo 2 concluído.
Passo 6/7: Gerando previsões finais híbridas...
Previsões híbridas geradas.
Passo 7/7: Avaliando o modelo Híbrido-Direto...

---
---
---
---
---
---
---
---
---
---
---
---
---
---
---
   Avaliação do Modelo HÍBRIDO-DIRETO de Chuva (Período de Teste)
---------------------------------------------

Horizonte de Previsão: chuva_t+1
  Média Real da Chuva: 22.99 mm
  MAE (Erro Médio Absoluto): 18.20 mm
  RMSE (Raiz do Erro Quadrático Médio): 28.33 mm

Horizonte de Previsão: chuva_t+2
  Média Real da Chuva: 22.41 mm
  MAE (Erro Médio Absoluto): 19.80 mm
  RMSE (Raiz do Erro Quadrático Médio): 27.41 mm

Horizonte de Previsão: chuva_t+3
  Média Real da Chuva: 21.72 mm
  MAE (Erro Médio Absoluto): 20.78 mm
  RMSE (Raiz do Erro Quadrático Médio): 26.66 mm

Horizonte de Previsão: chuva_t+4
  Média Real da Chuva: 21.21 mm
  

In [63]:
!pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.7/620.7 MB 6.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 37.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 35.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 39.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 33.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 38.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.9/71.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.2/323.2 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 39.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 8.3 MB/s eta 0:0

In [64]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler # <-- Para escalonar os dados
from statsmodels.tsa.deterministic import DeterministicProcess, CalendarFourier

# --- Importações do TensorFlow/Keras ---
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Garantir reprodutibilidade
tf.random.set_seed(42)

# --- 1. PREPARAÇÃO DOS DADOS DE CHUVA ---
print("Passo 1/9: Preparando dados...")
# (Esta seção não muda)
df_chuva = df_chuva.loc['2016-01-01':'2025-12-31'].copy()
df_semanal_chuva = df_chuva.resample('W').sum()
df_semanal_chuva['chuva'] = df_semanal_chuva['chuva'].fillna(0.0)

# --- 2. ENGENHARIA DE FEATURES (X) ---
print("Passo 2/9: Criando features...")
# (Esta seção não muda)
fourier_chuva = CalendarFourier(freq="A", order=4)
dp_chuva = DeterministicProcess(
    index=df_semanal_chuva.index,
    constant=True, order=1,
    additional_terms=[fourier_chuva], drop=True,
)
X_chuva_time = dp_chuva.in_sample()

def make_lags(ts, lags, name='y'):
    return pd.concat(
        {f'{name}_lag_{i}': ts.shift(i) for i in range(1, lags + 1)},
        axis=1)

X_chuva_lags = make_lags(df_semanal_chuva['chuva'], lags=8, name='chuva')
X_chuva_final = pd.concat([X_chuva_time, X_chuva_lags], axis=1)

# --- 3. CRIAÇÃO DOS ALVOS (Y) ---
print("Passo 3/9: Criando alvos...")
# (Esta seção não muda)
y_chuva_t1 = df_semanal_chuva['chuva'].shift(-1).rename('chuva_t+1')
y_chuva_t2 = df_semanal_chuva['chuva'].shift(-2).rename('chuva_t+2')
y_chuva_t3 = df_semanal_chuva['chuva'].shift(-3).rename('chuva_t+3')
y_chuva_t4 = df_semanal_chuva['chuva'].shift(-4).rename('chuva_t+4')
y_chuva_final = pd.concat([y_chuva_t1, y_chuva_t2, y_chuva_t3, y_chuva_t4], axis=1)

# --- 4. DIVISÃO DOS DADOS ---
print("Passo 4/9: Dividindo dados em treino/teste...")
# (Esta seção não muda, mas os nomes das variáveis são importantes)
df_model_data_chuva = pd.concat([X_chuva_final, y_chuva_final], axis=1)
df_model_data_chuva = df_model_data_chuva.dropna()

split_date = '2024-01-01'
df_train_chuva = df_model_data_chuva.loc[df_model_data_chuva.index < split_date]

target_cols = ['chuva_t+1', 'chuva_t+2', 'chuva_t+3', 'chuva_t+4']
X_train = df_train_chuva.drop(columns=target_cols)
y_train = df_train_chuva[target_cols]

X_test = X_chuva_final.loc[X_chuva_final.index >= split_date].copy()
y_test_truth = y_chuva_final.loc[X_test.index]

# --- 5. (NOVO) ESCALONAMENTO DOS DADOS ---
print("Passo 5/9: Escalonando dados para a Rede Neural...")

# Criar um escalonador para as features (X)
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

# Criar um escalonador para os alvos (y)
scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train)
# Não escalonamos y_test_truth, pois é o valor real para comparação

# --- 6. (NOVO) DEFINIÇÃO E TREINAMENTO DO MODELO DE REDE NEURAL (MIMO) ---
print("Passo 6/9: Definindo e treinando a Rede Neural...")

# Define a arquitetura da Rede Neural
nn_model_chuva = Sequential([
    # Camada de entrada (input_shape = número de features)
    Dense(64, activation='relu', input_shape=[X_train_scaled.shape[1]]),
    Dropout(0.2), # Dropout para prevenir overfitting
    Dense(32, activation='relu'),
    # Camada de saída: 4 neurônios (um para cada alvo: t+1, t+2, t+3, t+4)
    # Sem ativação (linear) pois é um problema de regressão
    Dense(4) 
])

# Compila o modelo
nn_model_chuva.compile(
    loss='mean_squared_error', # Função de perda para regressão
    optimizer='adam'           # Otimizador popular
)

# Define um 'Early Stopping' para parar o treino se o modelo não melhorar
early_stopping = EarlyStopping(
    monitor='val_loss', # Monitora a perda no set de validação
    patience=10,        # Número de épocas sem melhora antes de parar
    restore_best_weights=True # Restaura os pesos da melhor época
)

# Treina o modelo
history = nn_model_chuva.fit(
    X_train_scaled,
    y_train_scaled,
    epochs=100,           # Número máximo de épocas
    batch_size=16,        # Tamanho do lote
    validation_split=0.2, # Usa 20% dos dados de treino para validação
    callbacks=[early_stopping],
    verbose=0             # 0 = silencioso, 1 = barra de progresso
)

print("Treinamento da Rede Neural concluído.")

# --- 7. (NOVO) GERAR E REVERTER-ESCALA DAS PREVISÕES ---
print("Passo 7/9: Gerando e revertendo escala das previsões...")

# 1. Fazer previsões nos dados de teste escalonados
predictions_scaled = nn_model_chuva.predict(X_test_scaled)

# 2. REVERTER A ESCALA para trazer os dados de volta aos valores originais (mm de chuva)
predictions_array = scaler_y.inverse_transform(predictions_scaled)

# 3. Converter para o DataFrame final
df_chuva_forecasts = pd.DataFrame(
    predictions_array,
    index=X_test.index,
    columns=target_cols
)

# 4. Garantir que a chuva não seja negativa
df_chuva_forecasts[df_chuva_forecasts < 0] = 0

print("Previsões de chuva geradas.")

# --- 8. AVALIAÇÃO DO MODELO DE REDE NEURAL ---
print("Passo 8/9: Avaliando o modelo de Rede Neural...")
print("\n---" * 15)
print("   Avaliação do Modelo de REDE NEURAL (MIMO) de Chuva (Período de Teste)")
print("---" * 15)

for target in target_cols:
    preds_full = df_chuva_forecasts[target]
    truth_full = y_test_truth[target]
    
    truth_clean = truth_full.dropna()
    preds_clean = preds_full.loc[truth_clean.index]

    mae = mean_absolute_error(truth_clean, preds_clean)
    rmse = np.sqrt(mean_squared_error(truth_clean, preds_clean))
    mean_truth = truth_clean.mean()
    
    print(f"\nHorizonte de Previsão: {target}")
    print(f"  Média Real da Chuva: {mean_truth:.2f} mm")
    print(f"  MAE (Erro Médio Absoluto): {mae:.2f} mm")
    print(f"  RMSE (Raiz do Erro Quadrático Médio): {rmse:.2f} mm")

print("---" * 15)

# --- 9. AMOSTRA DAS PREVISÕES ---
print("Passo 9/9: Amostra das Previsões de Chuva Geradas (Head)")
print(df_chuva_forecasts.head())

2025-10-19 20:26:52.831778: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-10-19 20:26:52.952291: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-19 20:26:58.598275: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Passo 1/9: Preparando dados...
Passo 2/9: Criando features...
Passo 3/9: Criando alvos...
Passo 4/9: Dividindo dados em treino/teste...
Passo 5/9: Escalonando dados para a Rede Neural...
Passo 6/9: Definindo e treinando a Rede Neural...


/home/jefferson_correia/tcc/venv/lib/python3.12/site-packages/statsmodels/tsa/deterministic.py:569: FutureWarning: 'A' is deprecated and will be removed in a future version, please use 'YE' instead.
  index = pd.date_range("2020-01-01", freq=freq, periods=1)
/home/jefferson_correia/tcc/venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1760916418.911892    6028 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1760916418.917440    6028 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are ins

Treinamento da Rede Neural concluído.
Passo 7/9: Gerando e revertendo escala das previsões...
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
Previsões de chuva geradas.
Passo 8/9: Avaliando o modelo de Rede Neural...

---
---
---
---
---
---
---
---
---
---
---
---
---
---
---
   Avaliação do Modelo de REDE NEURAL (MIMO) de Chuva (Período de Teste)
---------------------------------------------

Horizonte de Previsão: chuva_t+1
  Média Real da Chuva: 22.99 mm
  MAE (Erro Médio Absoluto): 17.26 mm
  RMSE (Raiz do Erro Quadrático Médio): 25.27 mm

Horizonte de Previsão: chuva_t+2
  Média Real da Chuva: 22.41 mm
  MAE (Erro Médio Absoluto): 16.62 mm
  RMSE (Raiz do Erro Quadrático Médio): 25.26 mm

Horizonte de Previsão: chuva_t+3
  Média Real da Chuva: 21.72 mm
  MAE (Erro Médio Absoluto): 16.55 mm
  RMSE (Raiz do Erro Quadrático Médio): 25.64 mm

Horizonte de Previsão: chuva_t+4
  Média Real da Chuva: 21.21 mm
  MAE (Erro Médio Absoluto): 16.88 mm
  RMSE (Raiz do Erro Quadrático Médio): 25.38 mm


In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")

plt.rc(

    "figure",
    autolayout = True, 
    figsize = (11,4),
    titlesize = 18,
    titleweight = 'bold',

)

plt.rc(
    "axes",
    labelweight = 'bold',
    labelsize = "large",
    titleweight = "bold",
    titlesize = 16,
    titlepad = 10
    )

fig, ax = plt.subplots(figsize=(16,5))

ax.plot(df_semanal_chuva['Time'], df_semanal_chuva['chuva'], data=df_semanal_chuva, color= '0.75')
ax.plot (df_semanal_chuva['Time'], df_semanal_chuva['y_predc'], color='tab:blue', linewidth=3, label='Regressão Linear', zorder=3)

ax.set_title('Volume semanal de chuva (mm) no sistema cantareira entre 2016 e 2025')
ax.set_xlabel('Time')
ax.set_ylabel('Volume_util')

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
import matplotlib.pyplot as plt

decomposicao = seasonal_decompose(df_semanal_chuva['chuva'], model='additive', period=52)

fig, axes = plt.subplots(4, 1, sharex=True, figsize=(12, 10))


decomposicao.observed.plot(ax=axes[0], color='blue', legend=False)
axes[0].set_ylabel('Observado')


decomposicao.trend.plot(ax=axes[1], color='firebrick', legend=False)
axes[1].set_ylabel('Tendência')


decomposicao.seasonal.plot(ax=axes[2], color='green', legend=False)
axes[2].set_ylabel('Sazonalidade')


decomposicao.resid.plot(ax=axes[3], color='gray', legend=False, style='.')
axes[3].set_ylabel('Resíduo')


fig.suptitle('Decomposição da Série Temporal - Sistema Cantareira', fontsize=16)
plt.xlabel('Ano')
plt.tight_layout(rect=[0, 0.03, 1, 0.97]) # Ajusta o layout para o título principal não sobrepor os gráficos
plt.show()

### Fazendo a previsão da chuva para criar a variável do conjunto de teste do modelo de previsão do volume

In [ ]:
df_semanal_chuva['Time'] = np.arange(len(df_semanal_chuva.index))
df_semanal_chuva.head()

In [ ]:
# --- 0. IMPORTAÇÕES E CONFIGURAÇÃO ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.deterministic import DeterministicProcess, CalendarFourier
from xgboost import XGBRegressor

# --- 1. CARREGAMENTO, AGREGAÇÃO E ALINHAMENTO DE DADOS ---

# --- SUBSTITUA AQUI PELO CARREGAMENTO DOS SEUS DADOS ---
# Exemplo:
# dfv = pd.read_csv('seus_dados_de_volume.csv', parse_dates=['Date'], index_col='Date')
# df_chuva = pd.read_csv('seus_dados_de_chuva.csv', parse_dates=['Date'], index_col='Date')
# ---------------------------------------------------------

# Garantir a frequência diária antes de agregar
dfv = dfv.asfreq('D')
df_chuva = df_chuva.asfreq('D')

# Filtrar para o período de interesse (2016-2025)
df_periodo_vol = dfv.loc['2016-01-01':'2025-12-31'].copy()
df_periodo_chuva = df_chuva.loc['2016-01-01':'2025-12-31'].copy()

# Reamostrar para frequência semanal
# Volume: média da semana
df_semanal_volume = df_periodo_vol.resample('W').mean()
# Chuva: soma da semana
df_semanal_chuva = df_periodo_chuva.resample('W').sum()

# Juntar em um DataFrame final para garantir alinhamento
y1 = df_semanal_volume["Volume_util"].interpolate(method='linear')
chuva = df_semanal_chuva['chuva'].fillna(0.0)
df_final = pd.DataFrame({'Volume_util': y1, 'Chuva': chuva})


# --- 2. FEATURE ENGINEERING (MODELO 1: TENDÊNCIA E SAZONALIDADE) ---

fourier = CalendarFourier(freq="A", order=4)
dp = DeterministicProcess(
    index=df_final.index,
    constant=True,
    order=1,
    additional_terms=[fourier],
    drop=True,
)
x1 = dp.in_sample()


# --- 3. DIVISÃO EM DADOS DE TREINO E TESTE ---

split_date = '2024-01-01'

# Divisão para o Modelo 1
X_train = x1.loc[x1.index < split_date]
X_test = x1.loc[x1.index >= split_date]
y_train = df_final['Volume_util'].loc[df_final.index < split_date]
y_test = df_final['Volume_util'].loc[df_final.index >= split_date]


# --- 4. TREINAMENTO DO MODELO 1 E CÁLCULO DOS RESÍDUOS ---

model1 = LinearRegression()
model1.fit(X_train, y_train)

# Previsões da tendência e sazonalidade
y_pred_train = pd.Series(model1.predict(X_train), index=X_train.index)
y_pred_test = pd.Series(model1.predict(X_test), index=X_test.index)

# Cálculo dos resíduos
y_res_train = y_train - y_pred_train
y_res_test = y_test - y_pred_test
y_res = pd.concat([y_res_train, y_res_test])


# --- 5. FEATURE ENGINEERING (MODELO 2: RESÍDUOS E CHUVA) ---

def make_lags(ts, lags, name='y'):
    return pd.concat(
        {f'{name}_lag_{i}': ts.shift(i) for i in range(1, lags + 1)},
        axis=1)

# Lags dos resíduos
x2_residuos = make_lags(y_res, lags=2, name='res')

# Lags da chuva (variável exógena)
x2_chuva = make_lags(df_final['Chuva'], lags=4, name='chuva')

# Combinar todas as features para o Modelo 2
x2_final = pd.concat([x2_residuos, x2_chuva], axis=1)
x2_final = x2_final.fillna(0.0)

# Divisão das features do Modelo 2
X2_train = x2_final.loc[x2_final.index < split_date]
X2_test = x2_final.loc[x2_final.index >= split_date]


# --- 6. TREINAMENTO E PREVISÃO DO MODELO 2 (XGBOOST) ---

model2 = XGBRegressor(n_estimators=50, learning_rate=0.05, max_depth=5, random_state=0)
model2.fit(X2_train, y_res_train)

# Previsão dos resíduos no conjunto de treino
y_pred_res_train = pd.Series(model2.predict(X2_train), index=X2_train.index)

# Previsão RECURSIVA para os resíduos no conjunto de teste
X2_test_recursive = X2_test.copy()
y_pred_res_test_list = []

for index in X2_test_recursive.index:
    features_hoje = X2_test_recursive.loc[[index]]
    pred_hoje = model2.predict(features_hoje)[0]
    y_pred_res_test_list.append(pred_hoje)

    try:
        proximo_index = X2_test_recursive.index[X2_test_recursive.index.get_loc(index) + 1]
        # Atualiza o lag de resíduo para o próximo passo com a PREVISÃO de hoje
        X2_test_recursive.loc[proximo_index, 'res_lag_1'] = pred_hoje
        if 'res_lag_2' in X2_test_recursive.columns:
            X2_test_recursive.loc[proximo_index, 'res_lag_2'] = features_hoje['res_lag_1'].values[0]
    except IndexError:
        break # Fim do loop

y_pred_res_test = pd.Series(y_pred_res_test_list, index=X2_test.index)


# --- 7. COMBINAÇÃO FINAL E AVALIAÇÃO DO MODELO ---

# Previsão Híbrida Final
y_final_train = y_pred_train + y_pred_res_train
y_final_test = y_pred_test + y_pred_res_test

# Métricas de Avaliação
mae = mean_absolute_error(y_test, y_final_test)
rmse = np.sqrt(mean_squared_error(y_test, y_final_test))

print("---" * 15)
print("   Avaliação do Modelo Híbrido Semanal com Chuva (2024-2025)")
print("---" * 15)
print(f"MAE (Erro Médio Absoluto): {mae:.2f}")
print(f"RMSE (Raiz do Erro Quadrático Médio): {rmse:.2f}\n")
print(f"Interpretação: Em média, as previsões semanais do modelo erraram em {mae:.2f} pontos percentuais.")
print(f"Para referência, o volume médio no período de teste foi de {y_test.mean():.2f}%.")
print("---" * 15)


# --- 8. VISUALIZAÇÃO FINAL ---

fig, ax = plt.subplots(figsize=(14, 7))

df_final['Volume_util'].plot(ax=ax, color='black', alpha=0.7, style='-', label='Dados Reais (Média Semanal)')
y_final_train.plot(ax=ax, color='green', linestyle='--', label='Previsão Híbrida Semanal (Treino)')
y_final_test.plot(ax=ax, color='red', linestyle='-', label='Previsão Híbrida Semanal (Teste)')

ax.set_title('Previsão Híbrida Semanal com Chuva (2016-2025)')
ax.set_ylabel('Volume Médio Semanal (%)')
ax.legend()
plt.grid(True, which='both', linestyle='--', linewidth=0.5)
plt.show()